In [36]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms
from torch.utils.data import DataLoader


In [37]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))
])

In [38]:
train_dataset=datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
test_dataset=datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)
train_loader=DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)
test_loader=DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [39]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layer=nn.Sequential(
            nn.Conv2d(1,32,kernel_size=3,padding=1,),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

           
        )

        self.fc=nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )

    def forward(self,x):
        x=self.conv_layer(x)
        x=self.fc(x)
        return x






In [40]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = CNN().to(device)

In [41]:
model=CNN()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [42]:
epochs=5
for epoch in range(epochs):
    model.train()
    running_loss=0.0
    correct=0.0
    total=0.0

    for image,label in train_loader:
        optimizer.zero_grad()
        output=model(image)
        loss=criterion(output,label)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()
        _, predicted=torch.max(output,1)
        total+=label.size(0)
        correct+=(predicted==label).sum().item()

    print(
        f"Epoch {epoch+1}/{epochs}, "
        f"Loss: {running_loss/len(train_loader):.4f}, "
        f"Accuracy: {100*correct/total:.2f}%"
    )




Epoch 1/5, Loss: 0.1539, Accuracy: 95.29%
Epoch 2/5, Loss: 0.0442, Accuracy: 98.63%
Epoch 3/5, Loss: 0.0297, Accuracy: 99.09%
Epoch 4/5, Loss: 0.0216, Accuracy: 99.29%
Epoch 5/5, Loss: 0.0172, Accuracy: 99.43%


In [44]:
model.eval()
total=0
correct=0
for image,label in test_loader:
    output=model(image)
    _, predicted=torch.max(output,1)
    total+=label.size(0)
    correct+=(predicted==label).sum().item()
print(f"Test Accuracy: {100 * correct / total:.2f}%")

Test Accuracy: 98.95%
